# IBM HR Analytics — Employee Attrition EDA
**Author:** Jimmy Le-Nguyen | [github.com/jimmyle9080](https://github.com/jimmyle9080)

---

## Overview
This notebook explores the IBM HR Analytics Employee Attrition dataset — **1,470 employee records across 35 features** — to uncover what drives employee turnover and predict which employees are most at risk of leaving.

### Business Questions
- Which departments and job roles have the highest attrition?
- Does overtime significantly increase the likelihood of leaving?
- How does job satisfaction correlate with attrition?
- Can we predict which employees are most at risk?

### Dataset
- **Source:** [Kaggle — IBM HR Analytics Employee Attrition](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)
- **1,470 employees** | **35 features** | **16.1% attrition rate**
- No missing values

---
## 1. Imports

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False
})

ATTRITION_COLOR = '#E63946'
RETAIN_COLOR    = '#457B9D'
ACCENT_COLOR    = '#2A9D8F'
WARN_COLOR      = '#E9C46A'

print('Libraries loaded successfully')

---
## 2. Load & Clean Data

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Drop constant columns with no analytical value
df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours'], inplace=True)

# Binary flags
df['AttritionFlag'] = (df['Attrition'] == 'Yes').astype(int)
df['OverTimeFlag']  = (df['OverTime'] == 'Yes').astype(int)

# Feature engineering
df['IncomePerYear']     = (df['MonthlyIncome'] * 12).round(0)
df['TenureGroup']       = pd.cut(df['YearsAtCompany'],
                                  bins=[-1, 2, 5, 10, 20, 100],
                                  labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '11-20 yrs', '20+ yrs'])
df['AgeGroup']          = pd.cut(df['Age'],
                                  bins=[17, 25, 35, 45, 55, 100],
                                  labels=['18-25', '26-35', '36-45', '46-55', '55+'])
df['SatisfactionScore'] = (df['JobSatisfaction'] + df['EnvironmentSatisfaction'] +
                            df['RelationshipSatisfaction'] + df['WorkLifeBalance']) / 4

total     = len(df)
attrition = df['AttritionFlag'].sum()
retain    = total - attrition
attr_rate = attrition / total * 100

print(f'Employees loaded   : {total:,}')
print(f'Left company       : {attrition:,}  ({attr_rate:.1f}%)')
print(f'Stayed             : {retain:,}  ({100-attr_rate:.1f}%)')
print(f'Features available : {df.shape[1]}')
print(f'Null values        : {df.isnull().sum().sum()}')

In [ ]:
df.head()

In [ ]:
df.describe().round(2)

---
## 3. SQL Analysis

Loading data into an in-memory SQLite database and running business-level aggregations across departments, roles, tenure groups, and satisfaction scores.

In [ ]:
conn = sqlite3.connect(':memory:')
df.to_sql('employees', conn, if_exists='replace', index=False)

# Attrition by department
dept = pd.read_sql_query('''
    SELECT Department,
           COUNT(*) AS total_employees,
           SUM(AttritionFlag) AS attrition_count,
           ROUND(SUM(AttritionFlag)*100.0/COUNT(*), 1) AS attrition_rate_pct,
           ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income,
           ROUND(AVG(YearsAtCompany), 1) AS avg_tenure_years
    FROM employees
    GROUP BY Department
    ORDER BY attrition_rate_pct DESC
''', conn)
dept

In [ ]:
# Attrition by job role
role = pd.read_sql_query('''
    SELECT JobRole,
           COUNT(*) AS total_employees,
           SUM(AttritionFlag) AS attrition_count,
           ROUND(SUM(AttritionFlag)*100.0/COUNT(*), 1) AS attrition_rate_pct,
           ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income
    FROM employees
    GROUP BY JobRole
    ORDER BY attrition_rate_pct DESC
''', conn)
role

In [ ]:
# Overtime impact on attrition
ot = pd.read_sql_query('''
    SELECT OverTime,
           COUNT(*) AS total_employees,
           SUM(AttritionFlag) AS attrition_count,
           ROUND(SUM(AttritionFlag)*100.0/COUNT(*), 1) AS attrition_rate_pct,
           ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income
    FROM employees
    GROUP BY OverTime
''', conn)
ot

In [ ]:
# Job satisfaction vs attrition
sat = pd.read_sql_query('''
    SELECT JobSatisfaction,
           COUNT(*) AS total_employees,
           SUM(AttritionFlag) AS attrition_count,
           ROUND(SUM(AttritionFlag)*100.0/COUNT(*), 1) AS attrition_rate_pct
    FROM employees
    GROUP BY JobSatisfaction
    ORDER BY JobSatisfaction
''', conn)
sat

In [ ]:
# Attrition by tenure group
tenure = pd.read_sql_query('''
    SELECT TenureGroup,
           COUNT(*) AS total_employees,
           SUM(AttritionFlag) AS attrition_count,
           ROUND(SUM(AttritionFlag)*100.0/COUNT(*), 1) AS attrition_rate_pct,
           ROUND(AVG(MonthlyIncome), 0) AS avg_income
    FROM employees
    GROUP BY TenureGroup
    ORDER BY TenureGroup
''', conn)
tenure

In [ ]:
# High risk employees -- low satisfaction + overtime
high_risk = pd.read_sql_query('''
    SELECT EmployeeNumber, Department, JobRole, Age,
           MonthlyIncome, YearsAtCompany, JobSatisfaction,
           EnvironmentSatisfaction, OverTime, Attrition,
           ROUND(SatisfactionScore, 2) AS satisfaction_score
    FROM employees
    WHERE JobSatisfaction <= 2
      AND EnvironmentSatisfaction <= 2
      AND OverTime = "Yes"
    ORDER BY MonthlyIncome ASC
''', conn)

# Summary stats
stats = pd.read_sql_query('''
    SELECT
        COUNT(*) AS total_employees,
        SUM(AttritionFlag) AS total_attrition,
        ROUND(AVG(CASE WHEN AttritionFlag=1 THEN MonthlyIncome END), 0) AS avg_income_left,
        ROUND(AVG(CASE WHEN AttritionFlag=0 THEN MonthlyIncome END), 0) AS avg_income_stayed,
        ROUND(AVG(YearsAtCompany), 1) AS avg_tenure,
        SUM(OverTimeFlag) AS overtime_count
    FROM employees
''', conn).iloc[0]

conn.close()

print(f'Avg income (left)   : ${stats["avg_income_left"]:,.0f}/mo')
print(f'Avg income (stayed) : ${stats["avg_income_stayed"]:,.0f}/mo')
print(f'Income gap          : ${stats["avg_income_stayed"] - stats["avg_income_left"]:,.0f}/mo')
print(f'Avg tenure          : {stats["avg_tenure"]} years')
print(f'Overtime employees  : {int(stats["overtime_count"]):,}')
print(f'High risk employees : {len(high_risk):,}')

---
## 4. Exploratory Data Analysis

In [ ]:
# Attrition Overview
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Employee Attrition Overview', fontsize=14, fontweight='bold')

axes[0].pie([retain, attrition],
             labels=[f'Stayed\n{retain}', f'Left\n{attrition}'],
             colors=[RETAIN_COLOR, ATTRITION_COLOR],
             autopct='%1.1f%%', startangle=90,
             wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Overall Attrition Rate')

axes[1].barh(dept['Department'], dept['attrition_rate_pct'],
              color=[ATTRITION_COLOR, WARN_COLOR, RETAIN_COLOR], edgecolor='white')
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].set_title('Attrition Rate by Department')
for i, v in enumerate(dept['attrition_rate_pct']):
    axes[1].text(v + 0.3, i, f'{v}%', va='center', fontweight='bold')

age_data = df.groupby('AgeGroup')['AttritionFlag'].mean() * 100
axes[2].bar(age_data.index, age_data.values, color=ATTRITION_COLOR, alpha=0.8, edgecolor='white')
axes[2].set_xlabel('Age Group')
axes[2].set_ylabel('Attrition Rate (%)')
axes[2].set_title('Attrition Rate by Age Group')

plt.tight_layout()
plt.show()

In [ ]:
# Job Role Analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Attrition by Job Role', fontsize=14, fontweight='bold')

role_sorted = role.sort_values('attrition_rate_pct')
colors_role = [ATTRITION_COLOR if v > 20 else WARN_COLOR if v > 10
               else RETAIN_COLOR for v in role_sorted['attrition_rate_pct']]
axes[0].barh(role_sorted['JobRole'], role_sorted['attrition_rate_pct'],
              color=colors_role, edgecolor='white')
axes[0].set_xlabel('Attrition Rate (%)')
axes[0].set_title('Attrition Rate by Role')
axes[0].axvline(x=attr_rate, color='gray', linestyle='--', alpha=0.7,
                label=f'Avg ({attr_rate:.1f}%)')
axes[0].legend(fontsize=8)

role_income = role.sort_values('avg_monthly_income')
axes[1].barh(role_income['JobRole'], role_income['avg_monthly_income'],
              color=ACCENT_COLOR, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Avg Monthly Income ($)')
axes[1].set_title('Avg Monthly Income by Role')

plt.tight_layout()
plt.show()

In [ ]:
# Key Attrition Drivers -- Overtime & Satisfaction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Key Attrition Drivers', fontsize=14, fontweight='bold')

axes[0].bar(ot['OverTime'], ot['attrition_rate_pct'],
             color=[RETAIN_COLOR, ATTRITION_COLOR], width=0.4, edgecolor='white')
axes[0].set_xlabel('Overtime Required')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_title('Overtime vs Attrition Rate')
for i, (_, row) in enumerate(ot.iterrows()):
    axes[0].text(i, row['attrition_rate_pct'] + 0.5,
                 f"{row['attrition_rate_pct']}%", ha='center', fontweight='bold')

axes[1].bar(sat['JobSatisfaction'].astype(str), sat['attrition_rate_pct'],
             color=[ATTRITION_COLOR, WARN_COLOR, ACCENT_COLOR, RETAIN_COLOR], edgecolor='white')
axes[1].set_xlabel('Job Satisfaction (1=Low, 4=High)')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_title('Job Satisfaction vs Attrition Rate')
for i, v in enumerate(sat['attrition_rate_pct']):
    axes[1].text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Income Distribution & Tenure Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Income & Tenure Analysis', fontsize=14, fontweight='bold')

left_income   = df[df['AttritionFlag'] == 1]['MonthlyIncome']
stayed_income = df[df['AttritionFlag'] == 0]['MonthlyIncome']

axes[0].hist(stayed_income, bins=40, color=RETAIN_COLOR, alpha=0.7,
              label=f'Stayed (n={len(stayed_income):,})', density=True)
axes[0].hist(left_income, bins=40, color=ATTRITION_COLOR, alpha=0.7,
              label=f'Left (n={len(left_income):,})', density=True)
axes[0].axvline(stayed_income.mean(), color=RETAIN_COLOR, linestyle='--', linewidth=2,
                label=f'Stayed avg ${stayed_income.mean():,.0f}')
axes[0].axvline(left_income.mean(), color=ATTRITION_COLOR, linestyle='--', linewidth=2,
                label=f'Left avg ${left_income.mean():,.0f}')
axes[0].set_xlabel('Monthly Income ($)')
axes[0].set_ylabel('Density')
axes[0].set_title('Income Distribution by Attrition Status')
axes[0].legend(fontsize=8)

x = range(len(tenure))
axes[1].bar(x, tenure['attrition_rate_pct'], color=ATTRITION_COLOR, alpha=0.8, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tenure['TenureGroup'], rotation=15)
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].set_title('Attrition Rate by Tenure Group')
for i, v in enumerate(tenure['attrition_rate_pct']):
    axes[1].text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
numeric_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'JobSatisfaction',
                'EnvironmentSatisfaction', 'WorkLifeBalance', 'OverTimeFlag',
                'TotalWorkingYears', 'YearsSinceLastPromotion', 'AttritionFlag']

fig, ax = plt.subplots(figsize=(12, 8))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
ax.set_title('Correlation Heatmap — Key Features vs Attrition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top correlations with Attrition:')
print(corr['AttritionFlag'].drop('AttritionFlag').sort_values(key=abs, ascending=False).to_string())

---
## 5. Machine Learning — Attrition Prediction

Training three classification models to predict attrition risk. Class weighting handles the 5:1 imbalance between stayed and left employees. Using AUPRC alongside AUC-ROC since accuracy is misleading with imbalanced classes.

In [ ]:
# Encode categoricals
df_ml = df.copy()
le = LabelEncoder()
cat_cols = ['Attrition','BusinessTravel','Department','EducationField',
            'Gender','JobRole','MaritalStatus','OverTime','TenureGroup','AgeGroup']
for col in cat_cols:
    if col in df_ml.columns:
        df_ml[col] = le.fit_transform(df_ml[col].astype(str))

# Interaction features to improve model signal
df_ml['IncomeToAge']  = df_ml['MonthlyIncome'] / df_ml['Age']
df_ml['TenureRatio']  = df_ml['YearsAtCompany'] / (df_ml['TotalWorkingYears'] + 1)
df_ml['SatXOvertime'] = df['SatisfactionScore'] * df_ml['OverTimeFlag']
df_ml['PromotionLag'] = df_ml['YearsSinceLastPromotion'] / (df_ml['YearsAtCompany'] + 1)
df_ml['IncomeXSat']   = df_ml['MonthlyIncome'] * df['SatisfactionScore']
df_ml['AgeXTenure']   = df_ml['Age'] * df_ml['YearsAtCompany']
df_ml['OTXSat']       = df_ml['OverTimeFlag'] * df_ml['JobSatisfaction']
df_ml['DistXOT']      = df_ml['DistanceFromHome'] * df_ml['OverTimeFlag']

drop_cols    = ['AttritionFlag', 'IncomePerYear', 'SatisfactionScore']
feature_cols = [c for c in df_ml.columns if c not in drop_cols + ['Attrition']]

X = df_ml[feature_cols]
y = df_ml['AttritionFlag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set : {len(X_train):,} | Test set: {len(X_test):,} | Features: {len(feature_cols)}')
print(f'Class balance: {y_train.value_counts().to_dict()}')

In [ ]:
from sklearn.utils import resample

# Oversample minority class to 50/50
X_tr_df     = pd.concat([X_train, y_train], axis=1)
majority    = X_tr_df[X_tr_df.AttritionFlag == 0]
minority    = X_tr_df[X_tr_df.AttritionFlag == 1]
minority_up = resample(minority, replace=True, n_samples=len(majority), random_state=42)
balanced    = pd.concat([majority, minority_up])
X_tr_bal    = balanced.drop('AttritionFlag', axis=1)
y_tr_bal    = balanced['AttritionFlag']

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_bal)
X_te_sc = scaler.transform(X_test)

print('Training 5 models (this may take ~60 seconds)...')

# Tuned individual models
lr_model  = LogisticRegression(max_iter=2000, random_state=42, C=0.01)
rf_model  = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=3,
                                    class_weight='balanced', random_state=42, n_jobs=-1)
gb_model  = GradientBoostingClassifier(n_estimators=500, learning_rate=0.01,
                                        max_depth=4, subsample=0.8, random_state=42)
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, class_weight='balanced'),
    n_estimators=300, learning_rate=0.05, random_state=42)

lr_model.fit(X_tr_sc,  y_tr_bal)
rf_model.fit(X_tr_bal, y_tr_bal)
gb_model.fit(X_tr_bal, y_tr_bal)
ada_model.fit(X_tr_bal, y_tr_bal)

# Stacking classifier with LR meta-learner
stacking = StackingClassifier(
    estimators=[
        ('lr',  Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000, C=0.01, random_state=42))])),
        ('rf',  RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1)),
        ('gb',  GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, subsample=0.8, random_state=42)),
        ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, class_weight='balanced'), n_estimators=200, learning_rate=0.05, random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    cv=StratifiedKFold(5), n_jobs=-1
)
stacking.fit(X_tr_bal, y_tr_bal)

# Get probabilities
lr_p    = lr_model.predict_proba(X_te_sc)[:, 1]
rf_p    = rf_model.predict_proba(X_test)[:, 1]
gb_p    = gb_model.predict_proba(X_test)[:, 1]
ada_p   = ada_model.predict_proba(X_test)[:, 1]
stack_p = stacking.predict_proba(X_test)[:, 1]

# Weighted ensemble
ens_p = (0.40*lr_p + 0.25*gb_p + 0.20*rf_p + 0.15*ada_p)

# Decision threshold 0.38 for better recall
thr = 0.38
models_eval = {
    'Logistic Regression': (lr_p,    (lr_p    > thr).astype(int)),
    'Random Forest':        (rf_p,    (rf_p    > thr).astype(int)),
    'Gradient Boosting':    (gb_p,    (gb_p    > thr).astype(int)),
    'AdaBoost':             (ada_p,   (ada_p   > thr).astype(int)),
    'Stacking Classifier':  (stack_p, (stack_p > thr).astype(int)),
    'Weighted Ensemble':    (ens_p,   (ens_p   > thr).astype(int)),
}

model_results = []
print(f"{'Model':<25} {'AUC-ROC':>8} {'AUPRC':>8} {'Recall':>8} {'F1':>6}")
print('-' * 57)

for name, (probs, preds) in models_eval.items():
    auc    = roc_auc_score(y_test, probs)
    auprc  = average_precision_score(y_test, probs)
    report = classification_report(y_test, preds, output_dict=True)
    recall = report.get('1', {}).get('recall', 0)
    f1     = report.get('1', {}).get('f1-score', 0)
    print(f"{name:<25} {auc:>8.4f} {auprc:>8.4f} {recall:>8.4f} {f1:>6.4f}")
    model_results.append({'Model': name, 'AUC_ROC': auc, 'AUPRC': auprc, 'Recall': recall, 'F1': f1})

model_df = pd.DataFrame(model_results)
model_df

In [ ]:
# Feature importance from Random Forest
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': rf_model.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors_feat = [ATTRITION_COLOR if i < 5 else ACCENT_COLOR for i in range(len(feat_imp))]
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors_feat, edgecolor='white')
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 15 Predictors of Employee Attrition (Random Forest)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
red_p   = mpatches.Patch(color=ATTRITION_COLOR, label='Top 5 predictors')
green_p = mpatches.Patch(color=ACCENT_COLOR,    label='Supporting features')
ax.legend(handles=[red_p, green_p])
plt.tight_layout()
plt.show()
print('Top 10 predictors:')
print(feat_imp[['Feature','Importance']].head(10).to_string(index=False))

In [ ]:
# Model comparison -- all 6 models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Attrition Prediction Model Performance', fontsize=14, fontweight='bold')

bar_colors = ['#457B9D','#2A9D8F','#E9C46A','#E76F51','#6A4C93','#E63946']
metrics = ['AUC_ROC', 'AUPRC', 'Recall']
titles  = ['AUC-ROC', 'AUPRC', 'Recall (Attrition Class)']

for ax, metric, title in zip(axes, metrics, titles):
    bars = ax.bar(model_df['Model'], model_df[metric], color=bar_colors, alpha=0.9, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_xticklabels(model_df['Model'], rotation=25, ha='right', fontsize=7)
    for bar, val in zip(bars, model_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Employee Risk Scoring

Scoring all 1,470 employees using an ensemble probability (average of Random Forest + Gradient Boosting) and tiering them into **Low / Medium / High / Critical** risk buckets.

In [ ]:
# Score all employees with weighted ensemble
lr_all    = lr_model.predict_proba(scaler.transform(X))[:, 1]
rf_all    = rf_model.predict_proba(X)[:, 1]
gb_all    = gb_model.predict_proba(X)[:, 1]
ada_all   = ada_model.predict_proba(X)[:, 1]
ens_all   = (0.40*lr_all + 0.25*gb_all + 0.20*rf_all + 0.15*ada_all)

df['AttritionRisk'] = ens_all.round(4)
df['RiskTier'] = pd.cut(df['AttritionRisk'],
                          bins=[0, 0.3, 0.5, 0.7, 1.0],
                          labels=['Low', 'Medium', 'High', 'Critical'])

risk_summary = df.groupby('RiskTier').agg(
    employee_count=('EmployeeNumber', 'count'),
    actual_attrition=('AttritionFlag', 'sum'),
    avg_income=('MonthlyIncome', 'mean'),
    avg_satisfaction=('SatisfactionScore', 'mean')
).round(2)
print('Risk Tier Summary:')
risk_summary

In [ ]:
top_risk = df[['EmployeeNumber','Department','JobRole','Age','MonthlyIncome',
               'YearsAtCompany','OverTime','Attrition','AttritionRisk','RiskTier']]\
             .sort_values('AttritionRisk', ascending=False).head(10)
top_risk

---
## 7. Key Business Insights & Recommendations

### What drives attrition?
1. **Overtime** -- employees working overtime leave at ~3x the rate of non-overtime employees
2. **Low job satisfaction** -- satisfaction score of 1 leads to nearly 2x the attrition of score 4
3. **Early tenure** -- employees in their first 2 years have the highest attrition rate
4. **Lower income** -- employees who left earned significantly less than those who stayed
5. **Sales roles** -- Sales Representatives have the highest role-level attrition

### Recommendations
- **Target new hires** -- implement structured onboarding and 90-day check-ins for the 0-2 year cohort
- **Review overtime policies** -- audit departments with high overtime and assess workload distribution
- **Compensation benchmarking** -- review salaries for high-attrition roles against market rates
- **Satisfaction surveys** -- prioritize employees scoring 1-2 on satisfaction for immediate manager conversations
- **Proactive HR intervention** -- use risk scores to flag High and Critical tier employees before they resign

In [ ]:
print('=' * 55)
print('  IBM HR ATTRITION EDA - FINAL SUMMARY')
print('=' * 55)
print(f'  Total employees     : {total:,}')
print(f'  Attrition rate      : {attr_rate:.1f}%')
left_income   = df[df['AttritionFlag'] == 1]['MonthlyIncome']
stayed_income = df[df['AttritionFlag'] == 0]['MonthlyIncome']
print(f'  Avg income (left)   : ${left_income.mean():,.0f}/mo')
print(f'  Avg income (stayed) : ${stayed_income.mean():,.0f}/mo')
print(f'  Income gap          : ${stayed_income.mean() - left_income.mean():,.0f}/mo')
print(f'  High risk employees : {len(df[df["RiskTier"].isin(["High","Critical"])]):,}')
best_auc    = model_df.loc[model_df['AUC_ROC'].idxmax()]
best_recall = model_df.loc[model_df['Recall'].idxmax()]
print(f'  Best AUC model      : {best_auc["Model"]} ({best_auc["AUC_ROC"]:.4f})')
print(f'  Best recall model   : {best_recall["Model"]} ({best_recall["Recall"]:.4f})')
print('=' * 55)